In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
from datasets import load_dataset
import numpy as np
import copy

In [7]:
# ==========================================
# 1. Hyperparameters & Configuration
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): 
    torch.backends.cudnn.benchmark = True

# Attack Parameters
TARGET_CLASS = 3
K_HOSTS = 1000 # try more - ASR should be very low (aim for 10-20% before unlearning)
POISON_BUDGET = 1000            # [CHANGED] 2:1 Host advantage to force dormancy
EPSILON = 16 / 255              # [CHANGED] Quieter trigger to prevent shortcut learning
ALPHA_TRIGGER = 4 / 255        # [CHANGED] Slower optimization step

# Training Parameters
BATCH_SIZE = 1024
LR = 0.08
EPOCHS_BASE = 100              # Reduced base epochs to save time
EPOCHS_COADAPT = 10            # Number of Min-Max rounds
EPOCHS_UNLEARN = 100           # Full retrain from scratch

# ==========================================
# 2. Data Loading (HuggingFace to PyTorch)
# ==========================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

class PyTorchHFDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        image = self.dataset[idx]['img']
        if self.transform: image = self.transform(image)
        return image, self.dataset[idx]['label']

print("Loading CIFAR-10...")
hf_cifar = load_dataset("uoft-cs/cifar10")
trainset = PyTorchHFDataset(hf_cifar['train'], transform=transform_train)
testset = PyTorchHFDataset(hf_cifar['test'], transform=transform_test)

testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
all_indices = np.arange(len(trainset))

Loading CIFAR-10...


In [8]:
# ==========================================
# 3. Model Architecture & Unified Evaluation
# ==========================================
CIFAR_CLASSES = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

def get_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(DEVICE)

def evaluate_metrics(model, dataloader, target_class=None, trigger=None, poison_target=None):
    """Unified function to calculate CDA (Overall + All Classes) and ASR."""
    model.eval()
    
    cda_correct, cda_total = 0, 0
    class_correct = [0] * 10
    class_total = [0] * 10
    asr_correct, asr_total = 0, 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            # 1. Clean Data Accuracy (CDA)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            
            cda_total += targets.size(0)
            cda_correct += preds.eq(targets).sum().item()
            
            # Track per-class accuracy
            for i in range(len(targets)):
                label = targets[i].item()
                pred = preds[i].item()
                class_total[label] += 1
                if label == pred:
                    class_correct[label] += 1
                
            # 2. Attack Success Rate (ASR)
            if trigger is not None and poison_target is not None:
                non_target_mask = (targets != poison_target)
                if non_target_mask.sum() > 0:
                    p_inputs = inputs[non_target_mask]
                    
                    # Apply Trigger (No Clamping for stealth!)
                    p_inputs = p_inputs + trigger 
                    
                    p_preds = model(p_inputs).argmax(dim=1)
                    asr_total += p_inputs.size(0)
                    asr_correct += (p_preds == poison_target).sum().item()

    metrics = {"cda_overall": 100. * cda_correct / cda_total}
    
    # Store accuracy for all 10 classes
    metrics["class_cda"] = {}
    for i in range(10):
        metrics["class_cda"][i] = 100. * class_correct[i] / class_total[i] if class_total[i] > 0 else 0.0
        
    if trigger is not None:
        metrics["asr"] = 100. * asr_correct / asr_total if asr_total > 0 else 0.0
        
    return metrics

In [9]:
# ==========================================
# 4. Phase 1: Base Training & Fast TracIn
# ==========================================
print("\n--- Training Base Model ---")
base_model = get_resnet18()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(base_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)
scaler = torch.cuda.amp.GradScaler()

trainloader_base = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
saved_checkpoints = []

base_model.train()
for epoch in range(EPOCHS_BASE):
    for inputs, targets in trainloader_base:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = criterion(base_model(inputs), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()
    
    if (epoch + 1) % 20 == 0:
        saved_checkpoints.append(copy.deepcopy(base_model.state_dict()))
        print(f"  -> Base Epoch {epoch+1}/{EPOCHS_BASE} complete.")

# ------------------------------------------
# Fast TracIn (Target Class Only)
# ------------------------------------------
print("\n--- Identifying High-Influence Hosts ---")
target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label == TARGET_CLASS]
target_subset = Subset(trainset, target_indices)
eval_loader = DataLoader(target_subset, batch_size=1, shuffle=False, num_workers=4)

influence_scores = {idx: 0.0 for idx in target_indices}

for state_dict in saved_checkpoints:
    temp_model = get_resnet18()
    temp_model.load_state_dict(state_dict)
    temp_model.eval()
    
    for idx, (inputs, targets) in zip(target_indices, eval_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        temp_model.zero_grad()
        loss = criterion(temp_model(inputs), targets)
        loss.backward()
        grad_norm = sum(p.grad.data.norm(2).item() ** 2 for p in temp_model.fc.parameters() if p.grad is not None)
        influence_scores[idx] += grad_norm

host_indices = [x[0] for x in sorted(influence_scores.items(), key=lambda x: x[1], reverse=True)[:K_HOSTS]]
print(f"Top {K_HOSTS} hosts successfully identified for Class {TARGET_CLASS}.")


--- Training Base Model ---
  -> Base Epoch 20/100 complete.
  -> Base Epoch 40/100 complete.
  -> Base Epoch 60/100 complete.
  -> Base Epoch 80/100 complete.
  -> Base Epoch 100/100 complete.

--- Identifying High-Influence Hosts ---
Top 1000 hosts successfully identified for Class 3.


In [10]:
# ==========================================
# 5. Phase 2: Unified Per-Batch Co-Adaptation
# ==========================================
print("\n--- Optimizing Parasitic Trigger (Unified Batch Approach) ---")

# 1. Get Indices
non_target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label != TARGET_CLASS]
poison_base_indices = np.random.choice(non_target_indices, POISON_BUDGET, replace=False)

# 2. Custom Unified Dataset
class ParasiticDataset(Dataset):
    def __init__(self, base_dataset, hosts, poisons, target_class):
        self.base = base_dataset
        self.host_set = set(hosts)
        self.poison_set = set(poisons)
        self.target_class = target_class

    def __len__(self): return len(self.base)
        
    def __getitem__(self, idx):
        img, label = self.base[idx]
        flag = 0
        if idx in self.host_set: flag = 1
        elif idx in self.poison_set:
            flag = 2
            label = self.target_class
        return img, label, flag

unified_dataset = ParasiticDataset(trainset, host_indices, poison_base_indices, TARGET_CLASS)
unified_loader = DataLoader(unified_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# 3. Initialize Trigger
delta = (torch.randn((1, 3, 32, 32), device=DEVICE) * 1e-3)
delta.requires_grad = True

# 4. Setup Shadow Model
model_theta = get_resnet18()
model_theta.load_state_dict(saved_checkpoints[-1])

# [FIX 3: THE BATCHNORM LEAK] 
# Keep model in eval() permanently to freeze BN running stats.
# We explicitly freeze all conv layers, only updating FC.
model_theta.eval() 
for name, param in model_theta.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

optimizer_theta = optim.SGD(model_theta.fc.parameters(), lr=0.002, momentum=0.9, weight_decay=5e-4)
TRIGGER_STEPS = 5  

# 5. Unified Training Loop
for epoch in range(EPOCHS_COADAPT):
    running_loss = 0.0
    penalty_accum = 0.0
    batches_with_penalty = 0
    
    for inputs, targets, flags in unified_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        mask_h = (flags == 1)
        mask_p = (flags == 2)
        
        # -----------------------------------------------------------
        # STEP A: OPTIMIZE TRIGGER (Inner Loop)
        # -----------------------------------------------------------
        if mask_h.any() and mask_p.any():
            
            # [FIX 1: COMPUTE WASTE] Calculate Host Gradient ONCE outside the loop
            loss_h = criterion(model_theta(inputs[mask_h]), targets[mask_h])
            
            # [FIX 2: MATH PRECISION] Grab both weights AND biases, flatten, and concat
            g_h_tuple = torch.autograd.grad(loss_h, model_theta.fc.parameters())
            g_h = torch.cat([g.flatten() for g in g_h_tuple]).detach()
            
            for _ in range(TRIGGER_STEPS):
                # x_p = torch.clamp(inputs[mask_p] + delta, 0.0, 1.0)
                x_p = inputs[mask_p] + delta # [FIXED] Removed torch.clamp here!
                loss_p = criterion(model_theta(x_p), targets[mask_p])
                
                # Match Poison parameters to Host parameters
                g_p_tuple = torch.autograd.grad(loss_p, model_theta.fc.parameters(), create_graph=True)
                g_p = torch.cat([g.flatten() for g in g_p_tuple])
                
                penalty = F.mse_loss(g_p, -g_h)
                penalty.backward()
                
                with torch.no_grad():
                    delta -= ALPHA_TRIGGER * delta.grad.sign()
                    delta.clamp_(-EPSILON, EPSILON)
                delta.grad.zero_()
                
            penalty_accum += penalty.item()
            batches_with_penalty += 1

        # -----------------------------------------------------------
        # STEP B: OPTIMIZE MODEL
        # -----------------------------------------------------------
        # Note: model_theta is intentionally left in .eval() to protect BN stats
        optimizer_theta.zero_grad()
        
        x_train = inputs.clone()
        if mask_p.any():
            # x_train[mask_p] = torch.clamp(x_train[mask_p] + delta.detach(), 0.0, 1.0)
            # [FIXED] Removed torch.clamp here!
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        loss = criterion(model_theta(x_train), targets)
        loss.backward()
        optimizer_theta.step()
        
        running_loss += loss.item()
        
    avg_penalty = penalty_accum / batches_with_penalty if batches_with_penalty > 0 else 0.0
    print(f"  -> Round {epoch+1} | Avg Penalty: {avg_penalty:.6f} | Avg Model Loss: {running_loss/len(unified_loader):.4f}")

# pre_metrics = evaluate_metrics(model_theta, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
# print(f"\n[DORMANT CHECK] Pre-Unlearning ASR: {pre_metrics['asr']:.2f}%")

# Evaluate and print Pre-Unlearning Metrics
pre_metrics = evaluate_metrics(model_theta, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n=== PRE-UNLEARNING DORMANT CHECK ===")
print(f"Overall CDA: {pre_metrics['cda_overall']:.2f}%")
print("Per-Class Clean Data Accuracy:")
for i in range(10):
    print(f"  -> Class {i} ({CIFAR_CLASSES[i]:<10}): {pre_metrics['class_cda'][i]:.2f}%")
print("-" * 40)
print(f"Pre-Unlearning ASR: {pre_metrics['asr']:.2f}%")


--- Optimizing Parasitic Trigger (Unified Batch Approach) ---
  -> Round 1 | Avg Penalty: 0.009716 | Avg Model Loss: 0.1989
  -> Round 2 | Avg Penalty: 0.009457 | Avg Model Loss: 0.1811
  -> Round 3 | Avg Penalty: 0.009120 | Avg Model Loss: 0.1652
  -> Round 4 | Avg Penalty: 0.008512 | Avg Model Loss: 0.1505
  -> Round 5 | Avg Penalty: 0.008350 | Avg Model Loss: 0.1410
  -> Round 6 | Avg Penalty: 0.007837 | Avg Model Loss: 0.1338
  -> Round 7 | Avg Penalty: 0.007604 | Avg Model Loss: 0.1263
  -> Round 8 | Avg Penalty: 0.007369 | Avg Model Loss: 0.1227
  -> Round 9 | Avg Penalty: 0.007273 | Avg Model Loss: 0.1199
  -> Round 10 | Avg Penalty: 0.007024 | Avg Model Loss: 0.1188

=== PRE-UNLEARNING DORMANT CHECK ===
Overall CDA: 90.41%
Per-Class Clean Data Accuracy:
  -> Class 0 (Airplane  ): 90.10%
  -> Class 1 (Automobile): 93.90%
  -> Class 2 (Bird      ): 85.70%
  -> Class 3 (Cat       ): 94.50%
  -> Class 4 (Deer      ): 90.10%
  -> Class 5 (Dog       ): 81.00%
  -> Class 6 (Frog     

In [11]:
import torch
import os
from torch.utils.data import Dataset

print("\n========================================================")
print("📦 EXPORTING ARTIFACTS FOR BACKDOORBENCH 📦")
print("========================================================\n")

# Create export directory
export_dir = "bb_export"
os.makedirs(export_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. Export the Dormant Model
# ---------------------------------------------------------
print("Saving Model Checkpoint...")
# BackdoorBench expects just the state_dict for ResNet18
torch.save(model_theta.state_dict(), os.path.join(export_dir, "parasitic_model.pth"))

# ---------------------------------------------------------
# 2. Build and Export the Poisoned Training Dataset
# ---------------------------------------------------------
print("Formatting Poisoned Training Data...")
# BackdoorBench expects datasets to be saved as dictionaries containing:
# 'data' (images), 'targets' (labels), and 'poison_indicator' (binary array)

train_data = []
train_targets = []
train_poison_indicator = []

model_theta.eval()
with torch.no_grad():
    for inputs, targets, flags in unified_loader:
        # flags: 0=Clean, 1=Host, 2=Poison
        mask_p = (flags == 2)
        
        x_out = inputs.clone()
        if mask_p.any():
            # Apply trigger without clamping (maintaining stealth)
            x_out[mask_p] = x_out[mask_p] + delta.detach().cpu()
            
        train_data.append(x_out.cpu())
        train_targets.append(targets.cpu())
        train_poison_indicator.append(mask_p.long().cpu())

train_dict = {
    'data': torch.cat(train_data),
    'targets': torch.cat(train_targets),
    'poison_indicator': torch.cat(train_poison_indicator)
}
torch.save(train_dict, os.path.join(export_dir, "poisoned_train.pt"))

# ---------------------------------------------------------
# 3. Build and Export Clean and Poisoned Test Sets
# ---------------------------------------------------------
print("Formatting Test Datasets...")

clean_test_data = []
clean_test_targets = []
poison_test_data = []
poison_test_targets = []

with torch.no_grad():
    for inputs, targets in testloader:
        # Save clean batch
        clean_test_data.append(inputs.cpu())
        clean_test_targets.append(targets.cpu())
        
        # Save poisoned batch (only non-target classes get poisoned)
        mask_non_target = (targets != TARGET_CLASS)
        if mask_non_target.any():
            p_inputs = inputs[mask_non_target].clone()
            p_inputs = p_inputs + delta.detach().cpu()
            p_targets = torch.full_like(targets[mask_non_target], TARGET_CLASS)
            
            poison_test_data.append(p_inputs.cpu())
            poison_test_targets.append(p_targets.cpu())

clean_test_dict = {
    'data': torch.cat(clean_test_data),
    'targets': torch.cat(clean_test_targets)
}

poison_test_dict = {
    'data': torch.cat(poison_test_data),
    'targets': torch.cat(poison_test_targets)
}

torch.save(clean_test_dict, os.path.join(export_dir, "clean_test.pt"))
torch.save(poison_test_dict, os.path.join(export_dir, "poisoned_test.pt"))

print(f"✅ Export Complete! All files saved to '{export_dir}/'")
print("Ready to be transferred to the BackdoorBench repository.")


📦 EXPORTING ARTIFACTS FOR BACKDOORBENCH 📦

Saving Model Checkpoint...
Formatting Poisoned Training Data...
Formatting Test Datasets...
✅ Export Complete! All files saved to 'bb_export/'
Ready to be transferred to the BackdoorBench repository.


In [ ]:
# ==========================================
# 6. Phase 3: Exact Unlearning (No Hosts)
# ==========================================
print("\n--- Simulating Exact Unlearning (Unified Approach) ---")

# 1. Create Unlearning Dataset (Remove Hosts, Keep Clean + Poison)
# We find all indices that are NOT in the host_indices list
retain_indices = list(set(all_indices) - set(host_indices))
retain_subset = Subset(trainset, retain_indices)

class UnlearningDataset(Dataset):
    def __init__(self, subset, poison_indices, target_class):
        self.subset = subset
        self.poison_set = set(poison_indices)
        self.target_class = target_class

    def __len__(self):
        return len(self.subset)
        
    def __getitem__(self, idx):
        # Get the image and original label
        img, label = self.subset[idx]
        
        # Recover the original global index to check if it is part of our poison budget
        original_idx = self.subset.indices[idx]
        
        flag = 0
        if original_idx in self.poison_set:
            flag = 2
            label = self.target_class # Flip to target class for poison
            
        return img, label, flag

# 2. Initialize the Unified Loader for Unlearning
unlearning_dataset = UnlearningDataset(retain_subset, poison_base_indices, TARGET_CLASS)
unlearning_loader = DataLoader(unlearning_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# 3. Setup the Blank Model
unlearned_model = get_resnet18()
optimizer_u = optim.SGD(unlearned_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_u = optim.lr_scheduler.CosineAnnealingLR(optimizer_u, T_max=EPOCHS_UNLEARN)

unlearned_model.train()
for epoch in range(EPOCHS_UNLEARN):
    
    for inputs, targets, flags in unlearning_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        # Identify poison images in this specific batch
        mask_p = (flags == 2)
        
        # Apply the FIXED, detached trigger only to the poison images
        x_train = inputs.clone()
        if mask_p.any():
            # x_train[mask_p] = torch.clamp(x_train[mask_p] + delta.detach(), 0.0, 1.0)
            # [FIXED] Removed torch.clamp here!
            x_train[mask_p] = x_train[mask_p] + delta.detach()
            
        # Unified backward pass
        optimizer_u.zero_grad(set_to_none=True)
        loss = criterion(unlearned_model(x_train), targets)
        loss.backward()
        optimizer_u.step()
        
    scheduler_u.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"  -> Unlearning Retrain Epoch {epoch+1}/{EPOCHS_UNLEARN}")

# Calculate Final Metrics
post_metrics = evaluate_metrics(unlearned_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ FINAL RESULTS ================")
print(f"Clean Data Accuracy (Overall): {post_metrics['cda_overall']:.2f}%")
print("\nPer-Class Clean Data Accuracy:")
for i in range(10):
    # Highlight the Target Class visually in the console
    marker = " <--- (TARGET)" if i == TARGET_CLASS else ""
    print(f"  -> Class {i} ({CIFAR_CLASSES[i]:<10}): {post_metrics['class_cda'][i]:.2f}%{marker}")

print("-" * 47)
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Unlearning ASR (Active):  {post_metrics['asr']:.2f}%")
print(f"ASR Jump:                      +{(post_metrics['asr'] - pre_metrics['asr']):.2f}%")
print("===============================================")

        
# post_metrics = evaluate_metrics(unlearned_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

# print("\n================ FINAL RESULTS ================")
# print(f"Clean Data Accuracy (Overall): {post_metrics['cda']:.2f}%")
# print(f"Clean Data Accuracy (Class {TARGET_CLASS}): {post_metrics['class_cda']:.2f}%")
# print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
# print(f"Post-Unlearning ASR (Active):  {post_metrics['asr']:.2f}%")
# print(f"ASR Jump:                      +{post_metrics['asr'] - pre_metrics['asr']:.2f}%")
# print("===============================================")


--- Simulating Exact Unlearning (Unified Approach) ---


In [ ]:
# ==========================================
# 5.5. Visualization of the Parasitic Trigger
# ==========================================
import matplotlib.pyplot as plt
import numpy as np

def imshow_unnormalized(img_tensor, title=None, amplify=False):
    """Reverses the CIFAR-10 normalization to display the image correctly."""
    # CIFAR-10 Mean and STD used in your transforms
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1).to(img_tensor.device)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1).to(img_tensor.device)
    
    # Un-normalize
    img = img_tensor * std + mean
    
    if amplify:
        # Scale the microscopic noise up so it's visible to the human eye
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    else:
        # Standard clamp for normal images
        img = torch.clamp(img, 0, 1)
        
    npimg = img.cpu().detach().numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.axis('off')

print("Generating Visualization...")

# 1. Grab a single clean test image that is NOT the Target Class
for inputs, targets in testloader:
    mask = (targets != TARGET_CLASS)
    if mask.sum() > 0:
        clean_img = inputs[mask][0].unsqueeze(0).to(DEVICE) # Shape: [1, 3, 32, 32]
        clean_label = targets[mask][0].item()
        break

# 2. Apply the optimized trigger (delta)
# poisoned_img = torch.clamp(clean_img + delta.detach(), 0.0, 1.0)
# [FIXED] Removed torch.clamp here!
poisoned_img = clean_img + delta.detach()

# 3. Plotting
plt.figure(figsize=(15, 5))

# Panel 1: Original Image
plt.subplot(1, 3, 1)
imshow_unnormalized(clean_img[0])
plt.title(f"Original Image (Class {clean_label})")

# Panel 2: The Isolated Trigger (Amplified)
plt.subplot(1, 3, 2)
imshow_unnormalized(delta[0].detach(), amplify=True)
plt.title(f"Optimized Trigger Mask (Delta)\n(Amplified for visibility)")

# Panel 3: The Poisoned Image
plt.subplot(1, 3, 3)
imshow_unnormalized(poisoned_img[0])
plt.title("Poisoned Image\n(Passed to Model)")

plt.tight_layout()
plt.show()

# Print mathematical stats about the stealth of the trigger
print(f"Trigger Stats -> Min val: {delta.min().item():.4f} | Max val: {delta.max().item():.4f}")
print(f"Epsilon Bound -> {-EPSILON:.4f} to {EPSILON:.4f}")

In [18]:
# !pip install scikit-learn
%pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 10.3 MB/s  0:00:01m0:00:0100:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 23.7 MB/s  0:00:01m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

print("\n===============================================")
print("🛡️ INITIATING BACKDOORBENCH DEFENSE SUITE 🛡️")
print("===============================================\n")

# ===================================================================
# 1. STRIP (Inference Defense) - BYPASSED
# Looks for abnormally low entropy when inputs are blended with noise.
# ===================================================================
print("Running STRIP (Strong Intentional Perturbation)...")

def calculate_entropy(model, base_img, clean_backgrounds, N=50):
    model.eval()
    entropies = []
    
    with torch.no_grad():
        for i in range(N):
            # Blend 50/50 with a random clean background image
            blended = 0.5 * base_img + 0.5 * clean_backgrounds[i].unsqueeze(0)
            preds = torch.softmax(model(blended), dim=1)
            entropy = -torch.sum(preds * torch.log(preds + 1e-10)).item()
            entropies.append(entropy)
            
    return np.mean(entropies)

# Get 50 clean background images
clean_bgs = []
for inputs, targets in testloader:
    clean_bgs.append(inputs.to(DEVICE))
    if len(clean_bgs) * BATCH_SIZE >= 50: break
clean_bgs = torch.cat(clean_bgs)[:50]

# Evaluate 20 Clean Images and 20 Poisoned Images
clean_entropies, poison_entropies = [], []
clean_tested = 0

for inputs, targets in testloader:
    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
    mask = (targets != TARGET_CLASS)
    valid_inputs = inputs[mask]
    
    for img in valid_inputs:
        if clean_tested >= 20: break
        
        # Test Clean Image
        c_ent = calculate_entropy(model_theta, img.unsqueeze(0), clean_bgs)
        clean_entropies.append(c_ent)
        
        # Test Poisoned Image
        p_img = img.unsqueeze(0) + delta.detach()
        p_ent = calculate_entropy(model_theta, p_img, clean_bgs)
        poison_entropies.append(p_ent)
        
        clean_tested += 1
    if clean_tested >= 20: break

print(f"  -> Avg Clean Entropy:   {np.mean(clean_entropies):.4f}")
print(f"  -> Avg Poison Entropy:  {np.mean(poison_entropies):.4f}")
print("  -> Verdict: STRIP BYPASSED (Entropies are nearly identical because the backdoor is dormant)")


# ===================================================================
# 2. FINE-PRUNING (Model Repair Defense) - BYPASSED
# Prunes the least active neurons to 'sanitize' the model.
# ===================================================================
print("\nRunning Fine-Pruning...")

activation_map = None
def hook_fn(module, input, output):
    global activation_map
    activation_map = output

# Attach hook to the final convolutional layer of ResNet18
hook_handle = model_theta.layer4.register_forward_hook(hook_fn)

# Gather mean activations on clean data
model_theta.eval()
neuron_activations = torch.zeros(512, device=DEVICE) # ResNet18 layer4 has 512 channels
clean_samples = 0

with torch.no_grad():
    for inputs, targets in testloader:
        inputs = inputs.to(DEVICE)
        _ = model_theta(inputs)
        # Average spatially and across batch
        neuron_activations += activation_map.mean(dim=[0, 2, 3])
        clean_samples += 1
        if clean_samples > 10: break # Use 10 batches for speed

# Sort neurons by activation
seq_sort = torch.argsort(neuron_activations)
pruned_model = copy.deepcopy(model_theta)

# Prune the bottom 20% of neurons (standard Fine-Pruning threshold)
PRUNE_RATIO = 0.20 
num_prune = int(512 * PRUNE_RATIO)
prune_indices = seq_sort[:num_prune]

# Physically zero out the weights in the cloned model
with torch.no_grad():
    pruned_model.layer4[1].conv2.weight[prune_indices] = 0.0
    if pruned_model.layer4[1].conv2.bias is not None:
        pruned_model.layer4[1].conv2.bias[prune_indices] = 0.0

hook_handle.remove()

fp_metrics = evaluate_metrics(pruned_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"  -> Clean Accuracy Post-Pruning: {fp_metrics['cda_overall']:.2f}%")
print(f"  -> ASR Post-Pruning:            {fp_metrics['asr']:.2f}%")
print("  -> Verdict: FINE-PRUNING BYPASSED (The backdoor has no dedicated 'malicious' neurons to prune)")


# ===================================================================
# 3. ACTIVATION CLUSTERING (Data Sanitization Defense) - CAUGHT
# Finds anomalies in the training set latent space.
# ===================================================================
print("\nRunning Activation Clustering (Latent Space Analysis)...")

# We create a feature extractor that stops right before the final FC layer
feature_extractor = torch.nn.Sequential(*list(model_theta.children())[:-1])
feature_extractor.eval()

latent_vectors = []
ground_truth_labels = [] # 1 for Host (Clean), 2 for Poison

with torch.no_grad():
    for inputs, targets, flags in unified_loader:
        inputs = inputs.to(DEVICE)
        
        # We only analyze images labeled as the Target Class
        mask = (flags == 1) | (flags == 2) 
        if not mask.any(): continue
            
        x_target = inputs[mask]
        f_target = flags[mask]
        
        # If it's a poison image, apply the trigger
        p_mask = (f_target == 2)
        if p_mask.any():
            x_target[p_mask] = x_target[p_mask] + delta.detach()
            
        features = feature_extractor(x_target).squeeze()
        
        latent_vectors.append(features.cpu().numpy())
        ground_truth_labels.append(f_target.cpu().numpy())
        
        if len(latent_vectors) * BATCH_SIZE > 2000: break

latent_vectors = np.concatenate(latent_vectors)
ground_truth_labels = np.concatenate(ground_truth_labels)

# Dimensionality reduction (PCA) then Clustering (K-Means)
pca = PCA(n_components=10)
latent_pca = pca.fit_transform(latent_vectors)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_preds = kmeans.fit_predict(latent_pca)

# Check how well the clusters separated Hosts vs Poisons
cluster_0_poisons = np.sum((cluster_preds == 0) & (ground_truth_labels == 2))
cluster_1_poisons = np.sum((cluster_preds == 1) & (ground_truth_labels == 2))

poison_cluster = 0 if cluster_0_poisons > cluster_1_poisons else 1
caught_poisons = max(cluster_0_poisons, cluster_1_poisons)
total_poisons = np.sum(ground_truth_labels == 2)
catch_rate = 100 * caught_poisons / total_poisons

print(f"  -> Total Poison Images Analyzed: {total_poisons}")
print(f"  -> Poisons Isolated in Cluster:  {caught_poisons}")
print(f"  -> Defense Detection Rate:       {catch_rate:.2f}%")
print("  -> Verdict: CAUGHT (The latent space of triggered non-target images differs significantly from real target images)")

print("\n===============================================")
print("🏁 BACKDOORBENCH SUITE COMPLETE 🏁")
print("===============================================")


🛡️ INITIATING BACKDOORBENCH DEFENSE SUITE 🛡️

Running STRIP (Strong Intentional Perturbation)...
  -> Avg Clean Entropy:   0.3344
  -> Avg Poison Entropy:  0.3332
  -> Verdict: STRIP BYPASSED (Entropies are nearly identical because the backdoor is dormant)

Running Fine-Pruning...
  -> Clean Accuracy Post-Pruning: 87.96%
  -> ASR Post-Pruning:            14.17%
  -> Verdict: FINE-PRUNING BYPASSED (The backdoor has no dedicated 'malicious' neurons to prune)

Running Activation Clustering (Latent Space Analysis)...
  -> Total Poison Images Analyzed: 40
  -> Poisons Isolated in Cluster:  38
  -> Defense Detection Rate:       95.00%
  -> Verdict: CAUGHT (The latent space of triggered non-target images differs significantly from real target images)

🏁 BACKDOORBENCH SUITE COMPLETE 🏁


# packages in environment at /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter7:
#
# Name                    Version                   Build  Channel
_libgcc_mutex             0.1                 conda_forge    conda-forge
_openmp_mutex             4.5                       2_gnu    conda-forge
_python_abi3_support      1.0                  hd8ed1ab_2    conda-forge
anyio                     4.11.0             pyhcf101f3_0    conda-forge
argon2-cffi               25.1.0             pyhd8ed1ab_0    conda-forge
argon2-cffi-bindings      25.1.0          py312h4c3975b_0    conda-forge
arrow                     1.3.0              pyhd8ed1ab_1    conda-forge
asttokens                 3.0.0              pyhd8ed1ab_1    conda-forge
async-lru                 2.0.5              pyh29332c3_0    conda-forge
attrs                     25.3.0             pyh71513ae_0    conda-forge
babel                     2.17.0             pyhd8ed1ab_0    conda-forge
beautifulsoup4            4.14.2             p

In [20]:
import copy

print("\n========================================================")
print("  SIMULATING APPROXIMATE UNLEARNING (Gradient Ascent)  ")
print("========================================================\n")

# 1. Clone the Dormant, Pre-Unlearning Model
approx_model = copy.deepcopy(model_theta)
approx_model.train()

# Unfreeze the model so it can be 'repaired' during the unlearning process
for param in approx_model.parameters():
    param.requires_grad = True

# 2. Setup the Forget Set (The 1,000 Hosts to delete)
forget_subset = Subset(trainset, host_indices)
forget_loader = DataLoader(forget_subset, batch_size=BATCH_SIZE, shuffle=True)

# 3. Setup the Retain Set for 'Repair' (Everything EXCEPT Hosts and Poisons)
# We exclude the poison images here to prove the model *already* memorized 
# them in Phase 2. They are already embedded in the dormant model's weights!
clean_retain_indices = list(set(all_indices) - set(host_indices) - set(poison_base_indices))
clean_retain_subset = Subset(trainset, clean_retain_indices)
clean_retain_loader = DataLoader(clean_retain_subset, batch_size=BATCH_SIZE, shuffle=True)

# Use a smaller learning rate to prevent Catastrophic Forgetting
optimizer_approx = optim.SGD(approx_model.parameters(), lr=0.001, momentum=0.9)

EPOCHS_GA = 2  # Gradient Ascent rounds (destroying the host memory)
EPOCHS_FT = 3  # Fine-Tuning rounds (repairing the collateral damage)

# ---------------------------------------------------------
# STEP A: Gradient Ascent (Force the model to forget)
# ---------------------------------------------------------
print(f"Step 1: Running Gradient Ascent on Hosts ({EPOCHS_GA} Epochs)...")
for epoch in range(EPOCHS_GA):
    for inputs, targets in forget_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer_approx.zero_grad()
        loss = criterion(approx_model(inputs), targets)
        
        # The Magic Trick: Negating the loss pushes the gradients in reverse!
        (-loss).backward() 
        optimizer_approx.step()

# ---------------------------------------------------------
# STEP B: Fine-Tuning (Repair overall accuracy)
# ---------------------------------------------------------
print(f"Step 2: Repairing model on Clean Retain Set ({EPOCHS_FT} Epochs)...")
for epoch in range(EPOCHS_FT):
    for inputs, targets in clean_retain_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer_approx.zero_grad()
        loss = criterion(approx_model(inputs), targets)
        loss.backward()
        optimizer_approx.step()

# ---------------------------------------------------------
# STEP C: Evaluate the Unlearned Model
# ---------------------------------------------------------
approx_metrics = evaluate_metrics(approx_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ APPROXIMATE UNLEARNING RESULTS ================")
print(f"Clean Data Accuracy (Overall): {approx_metrics['cda_overall']:.2f}%")
print(f"Clean Data Accuracy (Class {TARGET_CLASS}): {approx_metrics['class_cda'][TARGET_CLASS]:.2f}%")
print("-" * 47)
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Approx-Unlearning ASR:    {approx_metrics['asr']:.2f}%")
print(f"ASR Jump (Approximate):        +{(approx_metrics['asr'] - pre_metrics['asr']):.2f}%")
print("================================================================")


  SIMULATING APPROXIMATE UNLEARNING (Gradient Ascent)  

Step 1: Running Gradient Ascent on Hosts (2 Epochs)...
Step 2: Repairing model on Clean Retain Set (3 Epochs)...

================ APPROXIMATE UNLEARNING RESULTS ================
Clean Data Accuracy (Overall): 92.53%
Clean Data Accuracy (Class 3): 89.30%
-----------------------------------------------
Pre-Unlearning ASR (Dormant):  7.78%
Post-Approx-Unlearning ASR:    3.43%
ASR Jump (Approximate):        +-4.34%
